# Multi-patient Results analysis
This notebook loads all complete patient results and follows the Results chapter order: with 3.1 being stimulation, 3.2 being the controllability comparisons, and 3 being the seizure phases.

## Configuration

In [ ]:
from pathlib import Path
import os
import pickle
import warnings
from scipy.stats import hypergeom
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, mannwhitneyu
from IPython.display import display

%matplotlib inline
plt.close("all")

PROJECT_ROOT = Path("/home/lheijmans/Master_Thesis")
RESULTS_ROOT = PROJECT_ROOT / "Results"
PLOTS_ROOT = PROJECT_ROOT / "Plots" / "multi_patient_analysis"
ANALYSIS_ROOT = PROJECT_ROOT / "Results" / "MultiPatientAnalysis"

PLOTS_ROOT.mkdir(parents=True, exist_ok=True)
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

PATIENT_IDS = []

if not PATIENT_IDS:
    PATIENT_IDS = sorted(
        p.name for p in RESULTS_ROOT.iterdir()
        if p.is_dir()
        and (p / "Controllability" / "ding_regime_nodes.parquet").exists()
        and (p / "Controllability" / "kashima_regime_nodes.parquet").exists()
        and (p / "Combined" / "stim_vs_baseline_summary.parquet").exists()
    )

print("Patients:", PATIENT_IDS)
print("Results root:", RESULTS_ROOT)
print("Analysis root:", ANALYSIS_ROOT)
print("Plots root:", PLOTS_ROOT)

TOP10_N = 10
TOP_N = 20
TOP20_N = 20
PHASE_ORDER = ["full", "early", "middle", "late"]
PHASE_DTYPE = pd.CategoricalDtype(PHASE_ORDER, ordered=True)
REGIME_TO_PHASE = {-1: "full", 0: "early", 1: "middle", 2: "late"}
# Surgery outcome groups
SUCCESSFUL_SURGERY = [
    "pt01", "pt02", "pt03", "pt08", "pt13", "pt15", "pt16", "pt17",
    "ummc003", "ummc004", "ummc005", "ummc006", "ummc008", "ummc009",
    "jh105", "umf001",
]

UNSUCCESSFUL_SURGERY = [
    "pt06", "pt07", "pt10", "pt12", "pt14", "jh101", "jh103", "jh108",
]


## Shared helper functions

In [ ]:
def patient_dirs(patient_id):
    base = RESULTS_ROOT / patient_id
    return {"base": base,"prepared": base / "Prepared","controllability": base / "Controllability","combined": base / "Combined",
            "analysis": base / "Analysis","plots": PLOTS_ROOT / patient_id,}


def derive_phase_from_regime_name(series):
    s = series.astype(str).str.lower()
    out = pd.Series(index=series.index, dtype=object)
    out[s.str.contains("full", na=False)] = "full"
    out[s.str.contains("early", na=False)] = "early"
    out[s.str.contains("middle", na=False)] = "middle"
    out[s.str.contains("mid", na=False)] = "middle"
    out[s.str.contains("late", na=False)] = "late"
    return out


def add_or_fix_phase(df, patient_id, method_name):

    out = df.copy()
    phase = pd.Series(index=out.index, dtype=object)

    if "regime" in out.columns:
        numeric_regime = pd.to_numeric(out["regime"], errors="coerce")
        phase_from_regime = numeric_regime.map(REGIME_TO_PHASE)
        phase = phase.fillna(phase_from_regime)

    if "regime_name" in out.columns:
        phase_from_name = derive_phase_from_regime_name(out["regime_name"])
        phase = phase.fillna(phase_from_name)

    if "phase" in out.columns:
        old_phase = out["phase"].astype(str)
        old_phase = old_phase.where(old_phase.isin(PHASE_ORDER), np.nan)
        phase = phase.fillna(old_phase)

    out["phase"] = phase

    # Quick fix for old error in linear code
    unique_phase = set(out["phase"].dropna().astype(str).unique())
    if unique_phase <= {"full"} and "chunk_idx" in out.columns:
        chunk_ids = sorted(pd.to_numeric(out["chunk_idx"], errors="coerce").dropna().astype(int).unique())
        if "original_seizure_idx" in out.columns:
            n_seizures = out["original_seizure_idx"].nunique()
        else:
            n_seizures = 4

        if n_seizures > 0 and len(chunk_ids) == 4 * n_seizures:
            print(
                f"{patient_id} {method_name}: repairing phase labels from chunk order "
                "because all phases were marked full."
            )
            chunk_to_phase = {}
            chunk_to_seizure = {}
            for block_idx, ph in enumerate(PHASE_ORDER):
                for s in range(n_seizures):
                    chunk = block_idx * n_seizures + s
                    chunk_to_phase[chunk] = ph
                    chunk_to_seizure[chunk] = s

            ci = pd.to_numeric(out["chunk_idx"], errors="coerce").astype("Int64")
            out["phase"] = ci.map(chunk_to_phase)
            out["original_seizure_idx"] = ci.map(chunk_to_seizure)
            out["seizure_number"] = out["original_seizure_idx"] + 1

    bad = out["phase"].isna()
    if bad.any():
        display(out.loc[bad].head())
        raise ValueError(f"{patient_id} {method_name}: could not derive phase for {bad.sum()} rows.")

    out["phase"] = out["phase"].astype(PHASE_DTYPE)
    return out


def safe_spearman(x, y):
    data = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(data) < 3 or data["x"].nunique() < 2 or data["y"].nunique() < 2:
        return np.nan, np.nan, len(data)
    rho, p = spearmanr(data["x"], data["y"])
    return float(rho), float(p), int(len(data))


def jaccard_from_sets(a, b):
    a, b = set(a), set(b)
    if len(a | b) == 0:
        return np.nan
    return len(a & b) / len(a | b)


def top_channels(df, value_col, n=10, channel_col="channel", ascending=False):
    d = df.sort_values(value_col, ascending=ascending)
    return d.head(n)[channel_col].tolist()


def duration_calc(df):
    if "regime_start_time" in df.columns and "regime_end_time" in df.columns:
        return df["regime_end_time"].iloc[0] - df["regime_start_time"].iloc[0]
    return np.nan

from scipy.stats import hypergeom

def top_n_overlap_stats(top_a, top_b, universe_size):
    """Return observed, expected, excess top-N overlap, Jaccard, and a one-sided exact hypergeometric p-value."""
    top_a, top_b = set(top_a), set(top_b)
    N = int(universe_size)
    n_a, n_b = len(top_a), len(top_b)
    observed = len(top_a & top_b)

    if N <= 0 or n_a == 0 or n_b == 0:
        return {
            "top_overlap_n": np.nan,
            "expected_overlap_n": np.nan,
            "excess_overlap_n": np.nan,
            "jaccard_top_n": np.nan,
            "overlap_p": np.nan,
        }

    expected = n_a * n_b / N
    union = len(top_a | top_b)
    return {
        "top_overlap_n": observed,
        "expected_overlap_n": expected,
        "excess_overlap_n": observed - expected,
        "jaccard_top_n": observed / union if union else np.nan,
        "overlap_p": float(hypergeom.sf(observed - 1, N, n_a, n_b)),
    }


In [ ]:
def group_signflip_test(df, value_col, alternative="greater", n_perm=100000, seed=42):
    """One-sample sign-flip permutation test on patient-level mean effects."""
    x=(df.dropna(subset=[value_col]).groupby("patient_id")[value_col].mean().to_numpy(float))
    if len(x)==0: return {"n_patients":0,"mean_effect":np.nan,"p_group":np.nan}
    obs=x.mean(); rng=np.random.default_rng(seed)
    if len(x)<=16:
        masks=np.arange(2**len(x),dtype=np.uint64)[:,None]
        bits=(masks >> np.arange(len(x),dtype=np.uint64)) & 1
        null=(np.where(bits,1.,-1.)*x).mean(1)
    else:
        null=(rng.choice([-1.,1.],size=(n_perm,len(x)))*x).mean(1)
    if alternative=="greater": p=(np.sum(null>=obs)+1)/(len(null)+1)
    elif alternative=="less": p=(np.sum(null<=obs)+1)/(len(null)+1)
    else: p=(np.sum(np.abs(null)>=abs(obs))+1)/(len(null)+1)
    return {"n_patients":len(x),"mean_effect":obs,"p_group":float(p)}

def significance_count(df,p_col,by):
    return df.groupby(by,observed=True).agg(n_tests=(p_col,"count"),n_significant=(p_col,lambda x:(x<0.05).sum()),fraction_significant=(p_col,lambda x:(x<0.05).mean())).reset_index()

def save_csv(df,name):
    p=ANALYSIS_ROOT/f"{name}.csv"; df.to_csv(p,index=False); print("saved",p); return p

def soz_rank_effect(df,rank_col,method):
    rows=[]; keys=["patient_id","original_seizure_idx","seizure_number","phase"]
    for key,g in df.groupby(keys,observed=True):
        a=g.loc[g.is_soz,rank_col].dropna(); b=g.loc[~g.is_soz,rank_col].dropna()
        if len(a) and len(b): rows.append(dict(zip(keys,key))|{"method":method,"mean_soz_rank":a.mean(),"mean_nsoz_rank":b.mean(),"rank_advantage":b.mean()-a.mean()})
    return pd.DataFrame(rows)

def relation(left,right,lrank,rrank,lname,rname,top_n=20):
    keys=["patient_id","original_seizure_idx","seizure_number","phase"]
    m=left[keys+["channel",lrank]].merge(right[keys+["channel",rrank]],on=keys+["channel"],how="inner")
    rows=[]
    for key,g in m.groupby(keys,observed=True):
        rho,p,N=safe_spearman(g[lrank],g[rrank]); n=min(top_n,N)
        A=g.nsmallest(n,lrank).channel; B=g.nsmallest(n,rrank).channel
        rows.append(dict(zip(keys,key))|{"comparison":f"{lname} vs {rname}","spearman_rho":rho,"spearman_p":p,"n_common_channels":N,**top_n_overlap_stats(A,B,N)})
    return pd.DataFrame(rows)

def phase_stability(df,rank_col,method,top_n=20):
    rows=[]; keys=["patient_id","original_seizure_idx","seizure_number"]
    for key,g in df.groupby(keys,observed=True):
        for a,b in combinations(["early","middle","late"],2):
            x=g[g.phase.astype(str)==a][["channel",rank_col]]; y=g[g.phase.astype(str)==b][["channel",rank_col]]
            m=x.merge(y,on="channel",suffixes=("_a","_b")); N=len(m)
            if N<3: continue
            rho,p,_=safe_spearman(m[f"{rank_col}_a"],m[f"{rank_col}_b"]); n=min(top_n,N)
            A=m.nsmallest(n,f"{rank_col}_a").channel; B=m.nsmallest(n,f"{rank_col}_b").channel
            rows.append(dict(zip(keys,key))|{"method":method,"phase_a":a,"phase_b":b,"spearman_rho":rho,"spearman_p":p,"n_common_channels":N,**top_n_overlap_stats(A,B,N)})
    return pd.DataFrame(rows)

## load and align results

In [ ]:
all_data = {}
all_ding = []
all_kashima_raw = []
all_kashima_nodes = []
all_stim = []

for patient_id in PATIENT_IDS:
    dirs = patient_dirs(patient_id)
    dirs["analysis"].mkdir(parents=True, exist_ok=True)
    dirs["plots"].mkdir(parents=True, exist_ok=True)

    ctrl_dir = dirs["controllability"]
    comb_dir = dirs["combined"]

    ding_path = ctrl_dir / "ding_regime_nodes.parquet"
    kashima_path = ctrl_dir / "kashima_regime_nodes.parquet"
    stim_path = comb_dir / "stim_vs_baseline_summary.parquet"
    ding_pkl_path = ctrl_dir / "ding_regime_results.pkl"

    print("\n" + "=" * 80)
    print("Loading", patient_id)
    print("Ding:", ding_path.exists(), ding_path)
    print("Kashima:", kashima_path.exists(), kashima_path)
    print("Stim:", stim_path.exists(), stim_path)

    missing = [p for p in [ding_path, kashima_path, stim_path] if not p.exists()]
    if missing:
        print("Skipping", patient_id, "missing:", missing)
        continue

    ding = pd.read_parquet(ding_path)
    kashima_raw = pd.read_parquet(kashima_path)
    stim = pd.read_parquet(stim_path)

    ding = add_or_fix_phase(ding, patient_id, "Ding")
    kashima_raw = add_or_fix_phase(kashima_raw, patient_id, "Kashima")
    stim = add_or_fix_phase(stim, patient_id, "Stimulation")

    # Standardize data
    ding["patient_id"] = patient_id
    kashima_raw["patient_id"] = patient_id
    stim["patient_id"] = patient_id

    if "seizure_number" not in ding.columns:
        ding["seizure_number"] = ding["original_seizure_idx"] + 1
    if "seizure_number" not in kashima_raw.columns:
        kashima_raw["seizure_number"] = kashima_raw["original_seizure_idx"] + 1
    if "seizure_number" not in stim.columns:
        stim["seizure_number"] = stim["original_seizure_idx"] + 1

    if "stim_channel" in stim.columns and "channel" not in stim.columns:
        stim["channel"] = stim["stim_channel"]
    if "stim_node" in stim.columns and "node_idx" not in stim.columns:
        stim["node_idx"] = stim["stim_node"]

    # SOZ check
    if "is_soz" not in kashima_raw.columns:
        if "is_soz" in ding.columns:
            soz_map = ding[["patient_id", "channel", "is_soz"]].drop_duplicates()
            kashima_raw = kashima_raw.merge(soz_map, on=["patient_id", "channel"], how="left")
        else:
            kashima_raw["is_soz"] = False

    if "is_soz" not in stim.columns:
        if "is_soz" in ding.columns:
            soz_map = ding[["patient_id", "channel", "is_soz"]].drop_duplicates()
            stim = stim.merge(soz_map, on=["patient_id", "channel"], how="left")
        else:
            stim["is_soz"] = False

    kashima_group_cols = ["patient_id", "original_seizure_idx", "seizure_number", "phase", "regime","node_idx", "channel", "is_soz",]
    
    kashima_group_cols = [c for c in kashima_group_cols if c in kashima_raw.columns]
    kashima_nodes = (kashima_raw.groupby(kashima_group_cols, as_index=False, observed=True, dropna=False).agg(
            mean_kashima_score=("kashima_score", "mean"),
            median_kashima_score=("kashima_score", "median"),
            std_kashima_score=("kashima_score", "std"),
            n_kashima_windows=("kashima_score", "count"),))

    ding["ding_rank"] = (ding.groupby(["patient_id", "original_seizure_idx", "phase"], observed=True)["mean_controllability"].rank(ascending=False, method="average"))

    kashima_nodes["kashima_rank"] = (kashima_nodes.groupby(["patient_id", "original_seizure_idx", "phase"], observed=True)["mean_kashima_score"].rank(ascending=False, method="average"))

    stim["stim_rank"] = (stim.groupby(["patient_id", "original_seizure_idx", "phase"], observed=True)["reduced_n_switches"].rank(ascending=False, method="average"))  

    ding_pkl = None
    if ding_pkl_path.exists():
        try:
            with open(ding_pkl_path, "rb") as f:
                ding_pkl = pickle.load(f)
        except Exception as e:
            print("Could not load Ding pickle:", e)

    all_data[patient_id] = {
        "dirs": dirs,
        "ding": ding,
        "kashima_raw": kashima_raw,
        "kashima_nodes": kashima_nodes,
        "stim": stim,
        "ding_pkl": ding_pkl,
    }

    all_ding.append(ding)
    all_kashima_raw.append(kashima_raw)
    all_kashima_nodes.append(kashima_nodes)
    all_stim.append(stim)

    print("Ding:", ding.shape)
    print("Kashima raw:", kashima_raw.shape)
    print("Kashima node-level:", kashima_nodes.shape)
    print("Stimulation:", stim.shape)

if not all_data:
    raise RuntimeError("No complete patients loaded.")

ding_all_patients = pd.concat(all_ding, ignore_index=True)
kashima_raw_all_patients = pd.concat(all_kashima_raw, ignore_index=True)
kashima_nodes_all_patients = pd.concat(all_kashima_nodes, ignore_index=True)
stim_all_patients = pd.concat(all_stim, ignore_index=True)

print("\nLoaded patients:", list(all_data.keys()))
print("All Ding:", ding_all_patients.shape)
print("All Kashima nodes:", kashima_nodes_all_patients.shape)
print("All Stim:", stim_all_patients.shape)

## Full-seizure tables

In [ ]:
ding_full = ding_all_patients[ding_all_patients["phase"].astype(str) == "full"].copy()
kashima_full = kashima_nodes_all_patients[kashima_nodes_all_patients["phase"].astype(str) == "full"].copy()
stim_full = stim_all_patients[stim_all_patients["phase"].astype(str) == "full"].copy()

print("Full Ding:", ding_full.shape)
print("Full Kashima:", kashima_full.shape)
print("Full Stimulation:", stim_full.shape)

display(ding_full.head())
display(kashima_full.head())
display(stim_full.head())

## Enrichment

In [ ]:
def topn_soz_enrichment(df, rank_col, method_name, top_n=20):

    rows = []

    group_cols = ["patient_id","original_seizure_idx","seizure_number","phase",]

    for keys, g in df.groupby(group_cols, observed=True):

        patient_id, seizure_idx, seizure_number, phase = keys

        g = g.dropna(subset=[rank_col, "is_soz"]).copy()

        N = len(g)
        K = int(g["is_soz"].sum())

        # Cannot select more channels than actually exist
        n = min(top_n, N)

        top = g.nsmallest(n, rank_col)

        observed = int(top["is_soz"].sum())

        # Expected SOZ count under random selection
        expected = n * K / N if N > 0 else np.nan

        # Enrichment above random expectation
        excess = observed - expected

        # Fold enrichment
        fold_enrichment = (
            observed / expected
            if expected > 0
            else np.nan
        )

        p_value = hypergeom.sf(observed - 1,N,K,n,)

        rows.append({
            "patient_id": patient_id,
            "original_seizure_idx": seizure_idx,
            "seizure_number": seizure_number,
            "phase": phase,
            "method": method_name,

            "n_channels": N,
            "n_soz_total": K,
            "top_n": n,

            "n_soz_top_n": observed,
            "expected_soz_top_n": expected,
            "excess_soz_top_n": excess,
            "fold_enrichment": fold_enrichment,
            "p_enrichment": p_value,
        })

    return pd.DataFrame(rows)

ding_top20_enrichment = topn_soz_enrichment(ding_all_patients,"ding_rank","Ding",top_n=20,)

kashima_top20_enrichment = topn_soz_enrichment(kashima_nodes_all_patients,"kashima_rank","Kashima",top_n=20,)

stim_top20_enrichment = topn_soz_enrichment(stim_all_patients,"stim_rank","Stimulation",top_n=20,)

top20_enrichment_all = pd.concat([ding_top20_enrichment,kashima_top20_enrichment,stim_top20_enrichment,],ignore_index=True,)



## regime analysis skip the not full ones

In [ ]:
def complete_keys(df):
    z=df[df.phase.astype(str).isin(["early","middle","late"])].groupby(["patient_id","original_seizure_idx"])["phase"].agg(lambda x:set(x.astype(str)))
    return {k for k,v in z.items() if {"early","middle","late"}.issubset(v)}
keys3=complete_keys(ding_all_patients)&complete_keys(kashima_nodes_all_patients)&complete_keys(stim_all_patients)
def keep3(df):
    idx=pd.MultiIndex.from_frame(df[["patient_id","original_seizure_idx"]]); return df[idx.isin(keys3)].copy()
ding3=keep3(ding_all_patients); kash3=keep3(kashima_nodes_all_patients); stim3=keep3(stim_all_patients)
included3=pd.DataFrame(sorted(keys3),columns=["patient_id","original_seizure_idx"])
print("Three-regime seizures:",len(included3),"patients:",included3.patient_id.nunique())
display(included3.groupby("patient_id").size().rename("n_seizures").reset_index())
save_csv(included3,"included_three_regime_seizures")

# 3.1 Stimulation analysis
## 3.1.1 Example baseline and stimulation runs

In [ ]:
EXAMPLE_PATIENT = "ummc005"
EXAMPLE_CHUNK = 0

example_base_dir = RESULTS_ROOT / EXAMPLE_PATIENT / "Baseline"
example_stim_dir = RESULTS_ROOT / EXAMPLE_PATIENT / "Stim"

baseline_summary_path = example_base_dir / f"baseline_summary_chunk_{EXAMPLE_CHUNK:02d}.parquet"
stim_summary_paths = sorted(example_stim_dir.glob(f"stim_summary_chunk_{EXAMPLE_CHUNK:02d}_node_*.parquet"))

print("Baseline summary:", baseline_summary_path, baseline_summary_path.exists())
print("Stimulation summary files:", len(stim_summary_paths))

baseline_example = pd.read_parquet(baseline_summary_path) if baseline_summary_path.exists() else pd.DataFrame()
stim_example = pd.concat([pd.read_parquet(p).assign(source_file=p.name) for p in stim_summary_paths], ignore_index=True) if stim_summary_paths else pd.DataFrame()

print("\nBaseline run table")
display(baseline_example)
print("\nStimulation run table")
display(stim_example.head(50))

if not stim_example.empty:
    stim_example.to_csv(ANALYSIS_ROOT / f"example_{EXAMPLE_PATIENT}_chunk_{EXAMPLE_CHUNK:02d}_stimulation_runs.csv", index=False)
if not baseline_example.empty:
    baseline_example.to_csv(ANALYSIS_ROOT / f"example_{EXAMPLE_PATIENT}_chunk_{EXAMPLE_CHUNK:02d}_baseline_run.csv", index=False)


In [ ]:
if (not baseline_example.empty and not stim_example.empty and
        "mean_n_switched" in baseline_example.columns and "mean_n_switched" in stim_example.columns):

    plot_df = stim_example.copy()

    if "stim_node" not in plot_df.columns:
        plot_df["stim_node"] = plot_df["source_file"].str.extract(r"node_(\d+)", expand=False).astype(float)

    plot_df = plot_df.sort_values("mean_n_switched").head(20)
    baseline_level = baseline_example["mean_n_switched"].mean()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(plot_df["stim_node"].astype(str), plot_df["mean_n_switched"])
    ax.axhline(baseline_level, linestyle="--", label=f"Baseline mean = {baseline_level:.2f}")
    ax.set_xlabel("Stimulated node")
    ax.set_ylabel("Mean number of switched nodes")
    ax.set_title(f"{EXAMPLE_PATIENT}, chunk {EXAMPLE_CHUNK:02d}: 20 stimulation targets with lowest recruitment")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("mean_n_switched is not available in both summary tables; inspect the displayed columns above.")


In [ ]:
def _first_trajectory_record(obj):
    if isinstance(obj, (list, tuple)) and len(obj):
        return _first_trajectory_record(obj[0])
    if isinstance(obj, dict):
        for key in ["trajectories", "runs", "results"]:
            if key in obj and isinstance(obj[key], (list, tuple)) and len(obj[key]):
                return _first_trajectory_record(obj[key][0])
        return obj
    return None

def _trajectory_radius(record):
    if not isinstance(record, dict):
        return None, None
    if "R" in record:
        R = np.asarray(record["R"])
        return R, record.get("dt", None)
    for xk, yk in [("x", "y"), ("X", "Y")]:
        if xk in record and yk in record:
            X, Y = np.asarray(record[xk]), np.asarray(record[yk])
            return np.sqrt(X**2 + Y**2), record.get("dt", None)
    return None, None

def plot_saved_trajectory(path, title, max_nodes=8):
    with open(path, "rb") as f:
        obj = pickle.load(f)
    record = _first_trajectory_record(obj)
    R, dt = _trajectory_radius(record)
    if R is None:
        print(f"Could not automatically find R or x/y trajectories in {path.name}.")
        if isinstance(record, dict):
            print("Available keys:", list(record.keys()))
        return
    R = np.asarray(R)
    if R.ndim != 2:
        print(f"Trajectory array has shape {R.shape}; expected nodes x time.")
        return
    t = np.arange(R.shape[1]) * float(dt) if dt is not None else np.arange(R.shape[1])
    fig, ax = plt.subplots(figsize=(12, 5))
    for node in range(min(max_nodes, R.shape[0])):
        ax.plot(t, R[node], label=f"node {node}")
    ax.axhline(0.5, linestyle="--", label="Switch threshold r=0.5")
    ax.set_xlabel("Time (s)" if dt is not None else "Time step")
    ax.set_ylabel("r(t)")
    ax.set_title(title)
    ax.legend(ncol=2)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

baseline_traj_path = example_base_dir / f"baseline_trajectories_chunk_{EXAMPLE_CHUNK:02d}.pkl"
stim_traj_paths = sorted(example_stim_dir.glob(f"*traj*chunk_{EXAMPLE_CHUNK:02d}*.pkl"))

print("Baseline trajectory file:", baseline_traj_path if baseline_traj_path.exists() else "not found")
print("Stimulation trajectory files found:", len(stim_traj_paths))

if baseline_traj_path.exists():
    plot_saved_trajectory(baseline_traj_path, f"{EXAMPLE_PATIENT}: baseline trajectory, chunk {EXAMPLE_CHUNK:02d}")

if stim_traj_paths:
    plot_saved_trajectory(stim_traj_paths[0], f"{EXAMPLE_PATIENT}: stimulation trajectory, {stim_traj_paths[0].name}")
else:
    print("No saved stimulation trajectory pickle found for this chunk; the stimulation summary table/plot above is still available.")


## 3.1.2 Stimulation efficacy across the patients

In [ ]:
stim_effect=(stim_full.groupby(["patient_id","original_seizure_idx","seizure_number"],as_index=False).agg(mean_reduction=("reduced_n_switches","mean"),median_reduction=("reduced_n_switches","median"),best_reduction=("reduced_n_switches","max"),worst_reduction=("reduced_n_switches","min"),sd_reduction=("reduced_n_switches","std"),n_targets=("channel","nunique")))
display(stim_effect.describe())
patient_stim=stim_effect.groupby("patient_id",as_index=False).agg(mean_reduction=("mean_reduction","mean"),mean_best_reduction=("best_reduction","mean"),n_seizures=("original_seizure_idx","nunique"))
display(patient_stim); save_csv(stim_effect,"3_1_2_stimulation_effect_per_seizure")
fig,ax=plt.subplots(figsize=(12,5)); d=patient_stim.sort_values("mean_best_reduction",ascending=False); ax.bar(d.patient_id,d.mean_best_reduction); ax.set(ylabel="Mean best reduction in switched nodes",title="Best stimulation effect per patient"); plt.xticks(rotation=60); plt.tight_layout(); plt.show()

## 3.1.3 Effective stimulation targets and soz

In [ ]:
stim_rank_soz=soz_rank_effect(stim_full,"stim_rank","Stimulation"); stim_enrich=topn_soz_enrichment(stim_full,"stim_rank","Stimulation",20)
stim_rank_soz=stim_rank_soz[stim_rank_soz.patient_id.isin(SUCCESSFUL_SURGERY)]; stim_enrich=stim_enrich[stim_enrich.patient_id.isin(SUCCESSFUL_SURGERY)]
display(significance_count(stim_enrich,"p_enrichment",["method"])); print("Cohort rank advantage:",group_signflip_test(stim_rank_soz,"rank_advantage")); print("Cohort excess SOZ in top20:",group_signflip_test(stim_enrich,"excess_soz_top_n")); save_csv(stim_enrich,"3_1_3_stim_SOZ_enrichment_successful")
p=stim_enrich.groupby("patient_id",as_index=False).agg(observed=("n_soz_top_n","mean"),expected=("expected_soz_top_n","mean")); x=np.arange(len(p)); w=.38; fig,ax=plt.subplots(figsize=(13,5)); ax.bar(x-w/2,p.observed,w,label="Observed"); ax.bar(x+w/2,p.expected,w,label="Expected"); ax.set_xticks(x); ax.set_xticklabels(p.patient_id,rotation=60); ax.set_ylabel("SOZ nodes in top 20"); ax.legend(); plt.tight_layout(); plt.show()

# 3.2 Controllability analysess

## 3.2.1 Example top-10 vs bottom-10 nodes

In [ ]:
EXAMPLE_CTRL_PATIENT="ummc005" if "ummc005" in all_data else list(all_data)[0]
EXAMPLE_SEIZURE=int(ding_full.loc[ding_full.patient_id==EXAMPLE_CTRL_PATIENT,"original_seizure_idx"].iloc[0])
def top_bottom_plot(df,value,rank,method):
    g=df[(df.patient_id==EXAMPLE_CTRL_PATIENT)&(df.original_seizure_idx==EXAMPLE_SEIZURE)].dropna(subset=[value,rank]); q=pd.concat([g.nsmallest(10,rank).assign(group="Top 10"),g.nlargest(10,rank).assign(group="Bottom 10")]); q=q.sort_values(value,ascending=False); fig,ax=plt.subplots(figsize=(12,5)); ax.bar(q.channel,q[value]); ax.set(title=f"{method}: top 10 vs bottom 10",ylabel=value); plt.xticks(rotation=60); plt.tight_layout(); plt.show(); display(q[["channel","is_soz",rank,value,"group"]])
top_bottom_plot(ding_full,"mean_controllability","ding_rank","Ding"); top_bottom_plot(kashima_full,"mean_kashima_score","kashima_rank","Kashima")

## 3.2.2 Ding & Kashima compared

In [ ]:
dk=relation(ding_full,kashima_full,"ding_rank","kashima_rank","Ding","Kashima",20); display(dk); display(significance_count(dk,"spearman_p",["comparison"])); display(significance_count(dk,"overlap_p",["comparison"])); print("Cohort rho:",group_signflip_test(dk,"spearman_rho")); print("Cohort excess overlap:",group_signflip_test(dk,"excess_overlap_n")); save_csv(dk,"3_2_2_Ding_Kashima_per_seizure")

## 3.2.3 Controllability and soz comparisons

In [ ]:
ctrl_rank_soz=pd.concat([soz_rank_effect(ding_full,"ding_rank","Ding"),soz_rank_effect(kashima_full,"kashima_rank","Kashima")]); ctrl_enrich=pd.concat([topn_soz_enrichment(ding_full,"ding_rank","Ding",20),topn_soz_enrichment(kashima_full,"kashima_rank","Kashima",20)]); ctrl_rank_soz=ctrl_rank_soz[ctrl_rank_soz.patient_id.isin(SUCCESSFUL_SURGERY)]; ctrl_enrich=ctrl_enrich[ctrl_enrich.patient_id.isin(SUCCESSFUL_SURGERY)]; display(significance_count(ctrl_enrich,"p_enrichment",["method"]));
for m,g in ctrl_enrich.groupby("method"): print(m,"top20 cohort:",group_signflip_test(g,"excess_soz_top_n"),"rank cohort:",group_signflip_test(ctrl_rank_soz[ctrl_rank_soz.method==m],"rank_advantage"))
save_csv(ctrl_enrich,"3_2_3_controllability_SOZ_enrichment_successful")

## 3.2.4 Controllability and stim targets

In [ ]:
ds=relation(ding_full,stim_full,"ding_rank","stim_rank","Ding","Stimulation",20); ks=relation(kashima_full,stim_full,"kashima_rank","stim_rank","Kashima","Stimulation",20); cs=pd.concat([ds,ks]); display(significance_count(cs,"spearman_p",["comparison"])); display(significance_count(cs,"overlap_p",["comparison"]));
for comp,g in cs.groupby("comparison"): print(comp,"rho",group_signflip_test(g,"spearman_rho"),"overlap",group_signflip_test(g,"excess_overlap_n"))
save_csv(cs,"3_2_4_controllability_vs_stimulation")

# 3.3 Seizure-regime analysis
Only seizures with early, middle and late regimes are included.
## 3.3.1 Stimulation ranking stability

In [ ]:
stim_stab=phase_stability(stim3,"stim_rank","Stimulation",20); display(significance_count(stim_stab,"spearman_p",["phase_a","phase_b"])); display(significance_count(stim_stab,"overlap_p",["phase_a","phase_b"])); save_csv(stim_stab,"3_3_1_stimulation_phase_stability")

## 3.3.2 Regime stim targets and SOZ

In [ ]:
s3=stim3[stim3.phase.astype(str).isin(["early","middle","late"])]; sr=soz_rank_effect(s3,"stim_rank","Stimulation"); se=topn_soz_enrichment(s3,"stim_rank","Stimulation",20); sr=sr[sr.patient_id.isin(SUCCESSFUL_SURGERY)]; se=se[se.patient_id.isin(SUCCESSFUL_SURGERY)]; display(significance_count(se,"p_enrichment",["phase"]));
for ph,g in se.groupby("phase",observed=True): print(ph,group_signflip_test(g,"excess_soz_top_n")); save_csv(se,"3_3_2_stim_phase_SOZ")

## 3.3.3 Regime vs full seizure stim rankings

In [ ]:
def phase_vs_full(df,rank,method):
 rows=[]; keys=["patient_id","original_seizure_idx","seizure_number"]
 for key,g in df.groupby(keys,observed=True):
  f=g[g.phase.astype(str)=="full"][["channel",rank]]
  for ph in ["early","middle","late"]:
   q=f.merge(g[g.phase.astype(str)==ph][["channel",rank]],on="channel",suffixes=("_full","_phase")); N=len(q)
   if N<3: continue
   rho,p,_=safe_spearman(q[f"{rank}_full"],q[f"{rank}_phase"]); n=min(20,N); A=q.nsmallest(n,f"{rank}_full").channel; B=q.nsmallest(n,f"{rank}_phase").channel; rows.append(dict(zip(keys,key))|{"method":method,"phase":ph,"spearman_rho":rho,"spearman_p":p,**top_n_overlap_stats(A,B,N)})
 return pd.DataFrame(rows)
stim_pf=phase_vs_full(stim3,"stim_rank","Stimulation"); display(stim_pf.groupby("phase").agg(mean_rho=("spearman_rho","mean"),mean_overlap=("top_overlap_n","mean"),expected=("expected_overlap_n","mean"),excess=("excess_overlap_n","mean"))); save_csv(stim_pf,"3_3_3_stim_phase_vs_full")

## 3.3.4 Controllability magnitude change and ranking stability

In [ ]:
def phase_magnitude(df,value,method):
 s=df[df.phase.astype(str).isin(["early","middle","late"])].groupby(["patient_id","original_seizure_idx","seizure_number","phase"],as_index=False,observed=True).agg(mean_value=(value,"mean")); e=s[s.phase.astype(str)=="early"][["patient_id","original_seizure_idx","mean_value"]].rename(columns={"mean_value":"early"}); s=s.merge(e,on=["patient_id","original_seizure_idx"]); s["percent_of_early"]=100*s.mean_value/s.early; s["method"]=method; return s
mag=pd.concat([phase_magnitude(ding3,"mean_controllability","Ding"),phase_magnitude(kash3,"mean_kashima_score","Kashima")]); display(mag.groupby(["method","phase"],observed=True).agg(mean_percent=("percent_of_early","mean"),sd=("percent_of_early","std"))); save_csv(mag,"3_3_4_controllability_magnitude")
fig,ax=plt.subplots(figsize=(8,5));
for m,g in mag.groupby("method"): q=g.groupby("phase",observed=True).percent_of_early.mean().reindex(["early","middle","late"]); ax.plot(q.index,q.values,marker="o",label=m)
ax.axhline(100,ls="--"); ax.set(ylabel="Controllability (% of early)",xlabel="Phase"); ax.legend(); plt.tight_layout(); plt.show()
stab=pd.concat([phase_stability(ding3,"ding_rank","Ding",20),phase_stability(kash3,"kashima_rank","Kashima",20)]); display(stab.groupby(["method","phase_a","phase_b"]).agg(mean_rho=("spearman_rho","mean"),mean_overlap=("top_overlap_n","mean"),expected=("expected_overlap_n","mean"),excess=("excess_overlap_n","mean"))); save_csv(stab,"3_3_4_controllability_ranking_stability")

## 3.3.5 Regime-specific controllability and SOZ

In [ ]:
d3=ding3[ding3.phase.astype(str).isin(["early","middle","late"])]; k3=kash3[kash3.phase.astype(str).isin(["early","middle","late"])]; cr=pd.concat([soz_rank_effect(d3,"ding_rank","Ding"),soz_rank_effect(k3,"kashima_rank","Kashima")]); ce=pd.concat([topn_soz_enrichment(d3,"ding_rank","Ding",20),topn_soz_enrichment(k3,"kashima_rank","Kashima",20)]); ce=ce[ce.patient_id.isin(SUCCESSFUL_SURGERY)]; cr=cr[cr.patient_id.isin(SUCCESSFUL_SURGERY)]; display(significance_count(ce,"p_enrichment",["method","phase"])); save_csv(ce,"3_3_5_controllability_phase_SOZ")

## 3.3.6 Regime-specific controllability vs stimulation

In [1]:
cs_phase=pd.concat([relation(d3,s3,"ding_rank","stim_rank","Ding","Stimulation",20),relation(k3,s3,"kashima_rank","stim_rank","Kashima","Stimulation",20)]); display(cs_phase.groupby(["comparison","phase"],observed=True).agg(mean_rho=("spearman_rho","mean"),mean_overlap=("top_overlap_n","mean"),expected=("expected_overlap_n","mean"),excess=("excess_overlap_n","mean"))); display(significance_count(cs_phase,"spearman_p",["comparison","phase"])); display(significance_count(cs_phase,"overlap_p",["comparison","phase"])); save_csv(cs_phase,"3_3_6_controllability_vs_stimulation_by_phase")

NameError: name 'pd' is not defined